In [ ]:
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt
#import seaborn as sns
#from scipy.stats import mannwhitneyu
#from scipy.stats import spearmanr
#from scipy.stats import kendalltau
#from sklearn.feature_selection import SelectKBest, f_classif
#from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
#from sklearn.decomposition import PCA
from sklearn.tree import DecisionTreeClassifier
#from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb
from xgboost import XGBClassifier
from catboost import CatBoostClassifier, Pool
#from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
#from sklearn.metrics import precision_recall_curve
#import optuna
#from bayes_opt import BayesianOptimization

In [ ]:
onus_columns = [f'onus_attribute_{i}' for i in range(1, 49) if i not in [7, 43, 44, 45, 46, 47, 48]]
bureau_enquiry_columns = [f'bureau_enquiry_{i}' for i in range(1, 51)]
transaction_columns = [f'transaction_attribute_{i}' for i in range(1, 665)]
bureau_columns = [f'bureau_{i}' for i in range(1, 453)if i not in [148, 433, 434, 435, 436, 437, 438, 444, 445, 446, 447, 448, 449, 451]]

In [ ]:
columns = transaction_columns + bureau_columns + onus_columns + bureau_enquiry_columns

In [ ]:
X = pd.read_csv("C:/Users/tirth/OneDrive/Desktop/Convolve/Dev_data_to_be_shared 3/Dev_data_to_be_shared.csv", index_col = 'account_number')

In [ ]:
#30 features found using CatBoost feature importances, Mann Whitney U Test and ANOVA Test comparison

cols_for_training = ['bad_flag', 'onus_attribute_26', 'onus_attribute_32', 'onus_attribute_29', 'onus_attribute_2', 'onus_attribute_33',
                   'onus_attribute_38', 'onus_attribute_35', 'onus_attribute_39', 'onus_attribute_17', 'onus_attribute_41',
                   'bureau_452', 'bureau_439', 'bureau_450', 'bureau_441', 'bureau_108', 'bureau_14', 'bureau_enquiry_45',
                   'bureau_enquiry_35', 'bureau_enquiry_31', 'bureau_enquiry_33', 'bureau_enquiry_41', 'bureau_enquiry_43',
                   'transaction_attribute_660', 'transaction_attribute_659', 'transaction_attribute_658', 'transaction_attribute_230',
                   'transaction_attribute_229', 'transaction_attribute_228']

In [ ]:
X_copy = pd.DataFrame(X[cols_for_training])

In [ ]:
y_copy = X_copy['bad_flag']

X_test_copy = X_copy.drop('bad_flag', axis=1)                                   #Seperating target column from dataset

In [ ]:
for i in X_test_copy.columns:
    X_test_copy[i] = X_copy[i].fillna(-10)                                      #NaN value filling

In [ ]:
scaler = StandardScaler()

X_test_copy_scaled = pd.DataFrame(scaler.fit_transform(X_test_copy))
X_test_copy_scaled.columns = X_test_copy.columns                                #Standard Scaling

In [ ]:
dt_params = {
    'max_depth': 5,
    'min_samples_leaf': 2,
    'min_samples_split': 2,
    'class_weight': {0: 0.2, 1: 0.8},
    'random_state': 69,
}

xgb_params = {
        'eval_metric': 'auc',
        'reg_lambda':  1.006641015807597,
        'max_depth': 5,
        'eta': 0.12337363223754441,
        'tree_method': 'hist',
        'scale_pos_weight': 8.33,
        'n_estimators': 1000,
        'random_state': 69,
        'device': 'cuda'
}

lgb_params = {
        'max_depth': 3,
        'learning_rate': 0.01949,
        'n_estimators': 1000,
        'min_child_weight': 8.515,
        'subsample': 0.8382,
        'colsample_bytree': 0.9249,
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'random_state': 69,
        'scale_pos_weight': 8.33,
}

cat_params = {
        'eval_metric': 'AUC',
        'l2_leaf_reg': 9.328,
        'max_depth': 4,
        'eta': 0.2335,
        'n_estimators': 1000,
        'scale_pos_weight': 8.33,
        'task_type': 'GPU',
        'random_state': 69
}                                                                               #All of these hyperparameters were found by tuning using Optuna, code below for xgboost and CatBoost

In [ ]:
dt_model = DecisionTreeClassifier(**dt_params)
xgb_model = XGBClassifier(**xgb_params)
lgb_model = lgb.LGBMClassifier(**lgb_params)
cat_model = CatBoostClassifier(**cat_params)                                    #Defining Models

In [ ]:
dt_model.fit(X_test_copy_scaled, y_copy)
xgb_model.fit(X_test_copy_scaled, y_copy)
lgb_model.fit(X_test_copy_scaled, y_copy)
cat_model.fit(X_test_copy_scaled, y_copy)                                       #Training

In [ ]:
X_test = pd.read_csv("C:/Users/tirth/OneDrive/Desktop/Convolve/validation_data_to_be_shared 3/validation_data_to_be_shared.csv", index_col = 'account_number')

In [ ]:
cols_for_output = cols_for_training.remove('bad_flag')

X_test = pd.DataFrame(X_test[cols_for_output])

for i in X_test.columns:
    X_test[i] = X_test[i].fillna(-10)

X_test_scaled = pd.DataFrame(scaler.transform(X_test))
X_test_scaled.columns = X_test.columns                                          #Preprocessing test dataset

In [ ]:
dt_preds = dt_model.predict_proba(X_test_scaled)[:, 1]
xgb_preds = xgb_model.predict_proba(X_test_scaled)[:, 1]
lgb_preds = lgb_model.predict_proba(X_test_scaled)[:, 1]
cat_preds = cat_model.predict_proba(X_test_scaled)[:, 1]                        #Predicting on test dataset

In [ ]:
final_preds = (xgb_preds + cat_preds + dt_preds + lgb_preds)/4                  #Generating output file

output_final = pd.DataFrame({'account_number': X_test.index, 'bad_flag': final_preds})
output_final.to_csv("C:/Users/tirth/OneDrive/Desktop/Convolve/validation_data_to_be_shared 3/Final_probabilities.csv", index = False)

In [ ]:
#XgBoost hyperparameter optimization

'''
X_train, X_valid, y_train, y_valid = train_test_split(X_test_copy, y_copy, test_size = 0.2, random_state = 69)

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train))
X_valid_scaled = pd.DataFrame(scaler.transform(X_valid))

X_train_scaled.columns = X_train.columns
X_valid_scaled.columns = X_valid.columns

xgb_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state = 69)

xgb_valid_auc = []
xgb_models = []

for train_index, valid_index in xgb_skf.split(X_train_scaled, y_train):
    X_train_fold, X_valid_fold = X_train_scaled.iloc[train_index], X_train_scaled.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

    def objective(trial):

        xgb_param = {
        'eval_metric': 'auc',
        'reg_lambda': trial.suggest_float('lambda', 0.5, 5.0),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'eta': trial.suggest_float('eta', 0.08, 0.8),
        'tree_method': 'hist',
        'scale_pos_weight': 8.33,
        'n_estimators': 1000,
        'early_stopping_rounds': 100,
        'random_state': 69,
        'device': 'cuda'
    }

        model = XGBClassifier(**xgb_param)
        model.fit(X_train_fold, y_train_fold, eval_set = [(X_valid_fold, y_valid_fold)], verbose = False)
        fold_preds = model.predict_proba(X_valid_fold)[:, 1]
        score = roc_auc_score(y_valid_fold, fold_preds)
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=69)

    best_params = study.best_params
    best_score = study.best_value

    xgb_valid_auc.append(best_score)
    xgb_models.append(best_params)

best_xgb_index = xgb_valid_auc.index(max(xgb_valid_auc))
best_xgb_params = xgb_models[best_xgb_index]

xgb_model = XGBClassifier(**best_xgb_params)
xgb_model.fit(X_train, y_train, verbose=696)

xgb_preds = xgb_model.predict_proba(test)[:, 1]

fpr, tpr, _ = roc_curve(y_valid, xgb_preds)
roc_auc = roc_auc_score(y_valid, xgb_preds)

plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
'''

In [ ]:
#CatBoost hyperparameter optimization

'''

for i in X_train_scaled.columns:
    if X_train_scaled[i].dtype == 'float64':
        X_train_scaled[i] = X_train_scaled[i].astype('str')
        X_valid_scaled[i] = X_valid_scaled[i].astype('str')

cat_skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 69)

cat_valid_auc = []
cat_models = []

for train_index, valid_index in cat_skf.split(X_train_scaled, y_train):
    X_train_fold, X_valid_fold = X_train_scaled.iloc[train_index], X_train_scaled.iloc[valid_index]
    y_train_fold, y_valid_fold = y_train.iloc[train_index], y_train.iloc[valid_index]

    X_train_pool = Pool(X_train_fold, y_train_fold, cat_features = X_train_fold.columns.values)
    X_valid_pool = Pool(X_valid_fold, y_valid_fold, cat_features = X_valid_fold.columns.values)

    def objective(trial):

        cat_param = {
        'eval_metric': 'AUC',
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 0.1, 1.0),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'eta': trial.suggest_float('eta', 0.08, 0.8),
        'n_estimators': 1000,
        'early_stopping_rounds':100,
        'scale_pos_weight': 8.33,
        'task_type': 'GPU',
        'random_state': 69
    }

        model = CatBoostClassifier(**cat_param)
        model.fit(X_train_pool, eval_set = X_valid_pool, verbose = False)
        fold_preds = model.predict_proba(X_valid_fold)[:, 1]
        score = roc_auc_score(y_valid_fold, fold_preds)
        return score

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=69)

    best_params = study.best_params
    best_score = study.best_value

    cat_valid_auc.append(best_score)
    cat_models.append(best_params)

best_cat_index = cat_valid_auc.index(max(cat_valid_auc))
best_cat_params = cat_models[best_cat_index]

cat_model = CatBoostClassifier(**best_cat_params)
cat_model.fit(X_train_scaled, y_train, verbose=200)

cat_preds = cat_model.predict_proba(X_valid_scaled)[:, 1]

fpr, tpr, _ = roc_curve(y_valid, cat_preds)
roc_auc = roc_auc_score(y_valid, cat_preds)

plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random Guess")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()
'''

In [ ]:
#code to generate top 100 features using mann whitney U test for each subset (subsets were generated by a seperate code)
#here X is a subset of the training dataset (Dev_data_to_be_shared)

'''
for i in X.columns:
    zeros = list(X.loc[X['bad_flag'] == 0][i])
    ones = list(X.loc[X['bad_flag'] == 1][i])
    u_stat, p_value = mannwhitneyu(zeros, ones, nan_policy = 'omit')
    manwhitneyu_scores[i] = p_value

sorted_manwhitneyu = dict((sorted(manwhitneyu_scores.items(), key=lambda item: abs(item[1])))[: 100])
manwhitneyu_df = pd.DataFrame(sorted_manwhitneyu.items(), columns=['Col_name', 'p_value'])
'''

In [ ]:
#code for ANOVA test

'''
def get_anova_features(X, y):
        select_k_best = SelectKBest(score_func=f_classif, k=100)
        select_k_best.fit(X, y)
        selected_indices = select_k_best.get_support(indices=True)
        return [X.columns[i] for i in selected_indices]
'''